In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.preprocessing import MinMaxScaler
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import math



In [3]:
# ===============================
# 1. PySpark Data Loading & Cleaning
# ===============================
spark = SparkSession.builder \
    .appName("AppleStockPrice") \
    .getOrCreate()

df = spark.read.csv("hdfs:///user1/Apple1.csv", header=True, inferSchema=True)
df_clean = df.select("Date", "Close").na.drop()
df_clean = df_clean.withColumn("Date", to_date(col("Date")))
df_clean = df_clean.orderBy("Date")

# Convert to pandas
df_pd = df_clean.toPandas()


25/10/08 10:17:59 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [4]:

# ===============================
# 2. Train/Test Split and Scaling
# ===============================
train_size = int(len(df_pd) * 0.8)
train_data = df_pd['Close'].values[:train_size].reshape(-1,1)
test_data = df_pd['Close'].values[train_size:].reshape(-1,1)

scaler = MinMaxScaler(feature_range=(0,1))
train_scaled = scaler.fit_transform(train_data)
test_scaled = scaler.transform(test_data)

# Combine scaled data for sequence generation
full_scaled = np.vstack((train_scaled, test_scaled))

In [5]:
# ===============================
# 3. Define GRU Model
# ===============================
class GRUModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, num_layers=2, dropout=0.2):
        super(GRUModel, self).__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        out, _ = self.gru(x)
        out = self.fc(out[:, -1, :])
        return out


In [6]:
import itertools
# ===============================
# 4. Hyperparameter Grid
# ===============================
param_grid = {
    'hidden_size': [50, 100],
    'num_layers': [1, 2],
    'dropout': [0.2, 0.3],
    'learning_rate': [0.001, 0.0005],
    'seq_len': [40, 60, 90]
}
all_combinations = list(itertools.product(
    param_grid['hidden_size'],
    param_grid['num_layers'],
    param_grid['dropout'],
    param_grid['learning_rate'],
    param_grid['seq_len']
))


In [7]:



# ===============================
# 5. Hyperparameter Tuning
# ===============================
best_val_loss = float('inf')
best_params = None
best_model_state = None

for combo in all_combinations:
    hidden_size, num_layers, dropout, lr, seq_len = combo
    
    # Create sequences from train_scaled
    x_train, y_train = [], []
    for i in range(seq_len, len(train_scaled)):
        x_train.append(train_scaled[i-seq_len:i, 0])
        y_train.append(train_scaled[i, 0])
    x_train, y_train = np.array(x_train), np.array(y_train)
    x_train = x_train.reshape(x_train.shape[0], x_train.shape[1], 1)
    
    # Convert to tensors
    X_tensor = torch.tensor(x_train, dtype=torch.float32)
    y_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(-1)
    dataset = TensorDataset(X_tensor, y_tensor)
    
    # Train-validation split (90%-10%)
    val_size = int(0.1 * len(dataset))
    train_size_ = len(dataset) - val_size
    train_subset, val_subset = random_split(dataset, [train_size_, val_size])
    
    train_loader = DataLoader(train_subset, batch_size=32, shuffle=False)
    val_loader = DataLoader(val_subset, batch_size=32, shuffle=False)
    
    # Model, loss, optimizer
    model = GRUModel(input_size=1, hidden_size=hidden_size, num_layers=num_layers, dropout=dropout)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Early stopping
    epochs = 20
    patience = 5
    patience_counter = 0
    best_val_combo_loss = float('inf')
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            output = model(X_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_val, y_val in val_loader:
                val_pred = model(X_val)
                val_loss += criterion(val_pred, y_val).item()
        val_loss /= len(val_loader)
        
        if val_loss < best_val_combo_loss:
            best_val_combo_loss = val_loss
            patience_counter = 0
            temp_best_state = model.state_dict()
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
    
    # Update global best
    if best_val_combo_loss < best_val_loss:
        best_val_loss = best_val_combo_loss
        best_params = {
            'hidden_size': hidden_size,
            'num_layers': num_layers,
            'dropout': dropout,
            'learning_rate': lr,
            'seq_len': seq_len
        }
        best_model_state = temp_best_state

print("Best validation loss:", best_val_loss)
print("Best hyperparameters:", best_params)



/home/hduser/.local/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
/home/hduser/.local/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
/home/hduser/.local/lib/python3.12/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  warnings.warn(
25/10/08 11:07:41 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 783052 ms exceeds timeout 120000 ms
25/10/08 11:07:42 WARN SparkContext: Killing executors is not supported by cur

25/10/08 11:07:43 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:123)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:688)
	at org.apache.spark.storage.BlockManagerMasterE

25/10/08 11:07:44 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:80)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:642)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1240)
	at org.apache.spark.executor.Executor.$anonfun$heartbeater$1(Executor.scala:296)
	at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.java:23)
	at org.apache.spark.util.Utils$.logUncaughtExceptions(Utils.scala:1928)

25/10/08 11:07:44 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:123)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:688)
	at org.apache.spark.storage.BlockManagerMasterE

25/10/08 11:07:44 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:123)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:688)
	at org.apache.spark.storage.BlockManagerMasterE

...

	at org.apache.spark.rpc.netty.LocalNettyRpcCallContext.send(NettyRpcCallContext.scala:50)
	at org.apache.spark.rpc.netty.NettyRpcCallContext.reply(NettyRpcCallContext.scala:32)
	at org.apache.spark.rpc.netty.RpcEndpointVerifier$$anonfun$receiveAndReply$1.applyOrElse(RpcEndpointVerifier.scala:31)
	... 8 more
25/10/08 11:07:46 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.st

25/10/08 11:07:47 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:123)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:688)
	at org.apache.spark.storage.BlockManagerMasterE

	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:123)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:688)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$1(BlockManagerMasterEndpoint.scala:687)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$register(BlockManagerMasterEndpoint.scala:725)
	at org.apache.spark.storage.BlockManagerMasterEndpoint$$anonfun$receiveAndReply$1.applyOrElse(BlockManagerMasterEndpoint.scala:133)
	at org.apache.spark.rpc.netty.Inbox.$anonfun$process$1(Inbox.scala:103)
	at org.apache.spark.rpc.netty.Inbox.safelyCall(Inbox.scala:213)
	at org.apache.spark.rpc.netty.Inbox.process(Inbox.scala:100)
	at org.apache.spark.rpc.netty.MessageLoop.org$apache$spark$rpc$netty$MessageLoop$$receiveLoop(MessageLoop.scala:7

25/10/08 11:07:47 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:123)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:688)
	at org.apache.spark.storage.BlockManagerMasterE

Best validation loss: 0.0006500453455373645
Best hyperparameters: {'hidden_size': 100, 'num_layers': 1, 'dropout': 0.3, 'learning_rate': 0.001, 'seq_len': 90}


In [8]:
# ===============================
# 6. Prepare Train and Test Sequences with Best Parameters
# ===============================
seq_len = best_params['seq_len']

# Train sequences
x_train_seq, y_train_seq = [], []
for i in range(seq_len, len(train_scaled)):
    x_train_seq.append(train_scaled[i-seq_len:i, 0])
    y_train_seq.append(train_scaled[i, 0])
x_train_seq = np.array(x_train_seq).reshape(-1, seq_len, 1)
y_train_seq = np.array(y_train_seq).reshape(-1,1)

# Test sequences (use last seq_len from train to start)
x_test_seq, y_test_seq = [], []
combined = np.vstack((train_scaled[-seq_len:], test_scaled))
for i in range(seq_len, len(combined)):
    x_test_seq.append(combined[i-seq_len:i, 0])
    y_test_seq.append(combined[i, 0])
x_test_seq = np.array(x_test_seq).reshape(-1, seq_len, 1)
y_test_seq = np.array(y_test_seq).reshape(-1,1)

# Convert to tensors
X_train_tensor = torch.tensor(x_train_seq, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_seq, dtype=torch.float32)
X_test_tensor = torch.tensor(x_test_seq, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_seq, dtype=torch.float32)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)



In [9]:
# ===============================
# 7. Train Final Model with Best Hyperparameters
# ===============================
final_model = GRUModel(
    input_size=1,
    hidden_size=best_params['hidden_size'],
    num_layers=best_params['num_layers'],
    dropout=best_params['dropout']
)
final_model.load_state_dict(best_model_state)
final_model.eval()



LSTMModel(
  (lstm): LSTM(1, 100, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=100, out_features=1, bias=True)
)

In [10]:
# ===============================
# 8. Predictions and Evaluation
# ===============================
# Train predictions
train_preds = []
with torch.no_grad():
    for X_batch, _ in train_loader:
        train_preds.append(final_model(X_batch).numpy())
train_preds = np.vstack(train_preds)
train_preds_rescaled = scaler.inverse_transform(train_preds)
y_train_rescaled = scaler.inverse_transform(y_train_seq)

# Test predictions
test_preds = []
with torch.no_grad():
    for X_batch, _ in test_loader:
        test_preds.append(final_model(X_batch).numpy())
test_preds = np.vstack(test_preds)
test_preds_rescaled = scaler.inverse_transform(test_preds)
y_test_rescaled = scaler.inverse_transform(y_test_seq)

# Metrics
train_rmse = np.sqrt(mean_squared_error(y_train_rescaled, train_preds_rescaled))
train_mae = mean_absolute_error(y_train_rescaled, train_preds_rescaled)
train_r2 = r2_score(y_train_rescaled, train_preds_rescaled)

test_rmse = np.sqrt(mean_squared_error(y_test_rescaled, test_preds_rescaled))
test_mae = mean_absolute_error(y_test_rescaled, test_preds_rescaled)
test_r2 = r2_score(y_test_rescaled, test_preds_rescaled)

print(f"Train RMSE: {train_rmse:.4f}, MAE: {train_mae:.4f}, R²: {train_r2:.4f}")
print(f"Test RMSE: {test_rmse:.4f}, MAE: {test_mae:.4f}, R²: {test_r2:.4f}")


Train RMSE: 4.2114, MAE: 3.2624, R²: 0.9747
Test RMSE: 7.1302, MAE: 5.0670, R²: 0.8396


In [11]:

# Optional: plot predictions vs actual
plt.figure(figsize=(12,6))
plt.plot(df['Date'][train_size:], y_test_rescaled, label='Actual', color='blue')
plt.plot(df['Date'][train_size:], test_preds_rescaled, label='Predicted', color='red')
plt.title('Apple Stock Price Prediction - Test Set', fontsize=16)
plt.xlabel('Date', fontsize=14)
plt.ylabel('Close Price (USD)', fontsize=14)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


PySparkTypeError: [NOT_SAME_TYPE] Argument `startPos` and `length` should be the same type, got int and NoneType.

<Figure size 1200x600 with 0 Axes>